# Геолокация студий звукозаписи



<font color="pink">**Аннотация данной части**</font>

Моя задача была понять, где в Москве открывать студию звукозаписи. Я собрала и визуализировала на карте данные о 15 студиях звукозаписи в Москве, а также добавила музыкальные вузы и клубы. Анализ показал, что в районе Artplay и Гнесинки студий мало, хотя там много музыкантов. На основе этого я рекомендую открыть новую студию именно в этом районе.

Этот код подключает Google Drive, чтобы все файлы (CSV, карты) сохранялись туда и не пропадали после закрытия. Далее он создаёт папку project_varvara/data и переходит в неё, чтобы все данные складывались в одно место, устанавливает библиотеки, а затем проверяет, что всё работает.

In [11]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Создаём папку на Google Drive
os.makedirs('/content/drive/MyDrive/project_varvara/data', exist_ok=True)

# Переходим в неё
os.chdir('/content/drive/MyDrive/project_varvara')

# Проверяем, где мы сейчас
print("Текущая папка:", os.getcwd())

# Устанавливаем библиотеки
!pip install folium openpyxl

print("Всё работает!")
import pandas as pd
print("pandas version:", pd.__version__)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Текущая папка: /content/drive/MyDrive/project_varvara
Всё работает!
pandas version: 2.2.2


# Сбор студий звукозаписм

Я начала свою работу со сбора данных о студиях звукозаписи. Сначала я вручную выбрала 15 студий на сайте 2gis.ru и сделала таблицу из названия, адреса, рейтинга и номера телефона.

Следующим шагом я получила точные координаты этих студий с помощью кода. Координаты адресов, которые не дал мне код, я вбила вручную с помощью Yandex maps. Собрав все нужные данные, я сделала интерактивную карту с точками.

Данный код автоматически получает географические координаты для 15 студий звукозаписи в Москве. Он создаёт таблицу с названиями и адресами студий, затем для каждого адреса отправляет запрос к сервису Nominatim (OpenStreetMap), который ищет адрес и возвращает его координаты. Функция get_coordinates_osm формирует полный адрес с добавлением «Москва, Россия», отправляет запрос, обрабатывает ответ и извлекает координаты. Чтобы не перегружать сервис, между запросами делается пауза 1.5 секунды (time.sleep(1.5)). В конце код сохраняет полученную таблицу с координатами в файл studios_with_coords.csv и выводит результат на экран.

In [12]:
import pandas as pd
import requests
import time
import re

# Функция для получения координат
def get_coordinates_osm(address):
    full_address = f"{address}, Москва, Россия"

    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": full_address,
        "format": "json",
        "limit": 1
    }
    headers = {
        "User-Agent": "MoscowStudioProject/1.0"
    }

    try:
        response = requests.get(url, params=params, headers=headers)
        data = response.json()
        if data:
            lat = float(data[0]["lat"])
            lon = float(data[0]["lon"])
            print(f" {address[:40]}... → {lat:.4f}, {lon:.4f}")
            return lat, lon
        else:
            print(f" Не найдено: {address}")
            return None, None
    except Exception as e:
        print(f" Ошибка: {address}")
        return None, None

# Мои данные
data = {
    "name": [
        "Grusha Music", "Moon dragon", "KomorKing sound",
        "Продюсерский центр Игоря Сандлера", "Pioneer DJ School",
        "Хендрикс", "ЦАО Records", "Звук Во рту", "Вайб Рекордс",
        "333 sound", "Overtime", "Sandler Studio", "Major Studio",
        "All Music Studio", "Showbizrecords"
    ],
    "address": [
        "Большой Путинковский переулок, 5",
        "Петровский бульвар, 9 ст2",
        "Красный Октябрь, Болотная набережная, 3 ст2",
        "Потаповский переулок, 3 ст1",
        "Нижний Сусальный переулок, 5 ст4",
        "Марксистская улица, 34 к4",
        "Улица Земляной Вал, 50а ст3",
        "Улица Покровка, 2/1 ст2",
        "Нижняя Сыромятническая улица, 10 ст8",
        "Рождественский бульвар, 10/7 ст1",
        "Налесный переулок, 4",
        "Потаповский переулок, 3 ст1",
        "Нижняя Сыромятническая улица, 10 ст40",
        "Старокирочный переулок, 2",
        "Старая Басманная улица, 34"
    ]
}

df = pd.DataFrame(data)

# Получаем координаты
lat_list = []
lon_list = []

for address in df["address"]:
    lat, lon = get_coordinates_osm(address)
    lat_list.append(lat)
    lon_list.append(lon)
    time.sleep(1.5)

df["lat"] = lat_list
df["lon"] = lon_list

# Сохраняем
df.to_csv("studios_with_coords.csv", index=False)
print("\n Файл сохранён: studios_with_coords.csv")

print("\n Результат:")
print(df[["name", "address", "lat", "lon"]])

 Большой Путинковский переулок, 5... → 55.7672, 37.6077
 Не найдено: Петровский бульвар, 9 ст2
 Не найдено: Красный Октябрь, Болотная набережная, 3 ст2
 Потаповский переулок, 3 ст1... → 55.7618, 37.6400
 Не найдено: Нижний Сусальный переулок, 5 ст4
 Марксистская улица, 34 к4... → 55.7353, 37.6633
 Не найдено: Улица Земляной Вал, 50а ст3
 Улица Покровка, 2/1 ст2... → 55.7595, 37.6472
 Нижняя Сыромятническая улица, 10 ст8... → 55.7530, 37.6696
 Не найдено: Рождественский бульвар, 10/7 ст1
 Налесный переулок, 4... → 55.7778, 37.6821
 Потаповский переулок, 3 ст1... → 55.7618, 37.6400
 Не найдено: Нижняя Сыромятническая улица, 10 ст40
 Старокирочный переулок, 2... → 55.7681, 37.6840
 Старая Басманная улица, 34... → 55.7689, 37.6687

 Файл сохранён: studios_with_coords.csv

 Результат:
                                 name  \
0                        Grusha Music   
1                         Moon dragon   
2                     KomorKing sound   
3   Продюсерский центр Игоря Сандлера   
4   

Тут я уже вручную исправляю координаты для тех адресов, которые Nominatim не смог найти. Он загружает ранее созданный файл studios_with_coords.csv, где у некоторых студий координаты остались пустыми, и заменяет их на значения, которые я уточнила через Яндекс.Карты. Для каждого адреса из словаря manual_coords код находит соответствующую строку в таблице и обновляет колонки lat (широта) и lon (долгота).

In [13]:
import pandas as pd

df = pd.read_csv("studios_with_coords.csv")

# Уточнённые координаты
manual_coords = {
    "Петровский бульвар, 9 ст2": (55.768588, 37.614779),
    "Красный Октябрь, Болотная набережная, 3 ст2": (55.740000, 37.609505),
    "Нижний Сусальный переулок, 5 ст4": (55.760206, 37.663889),
    "Улица Земляной Вал, 50а ст3": (55.751148, 37.655248),
    "Рождественский бульвар, 10/7 ст1": (55.766223, 37.626196),
    "Нижняя Сыромятническая улица, 10 ст40": (55.752794, 37.669782)
}

# Обновляем координаты в таблице
for address, (lat, lon) in manual_coords.items():
    mask = df["address"] == address
    df.loc[mask, "lat"] = lat
    df.loc[mask, "lon"] = lon
    print(f"Обновлено: {address[:40]}... → {lat}, {lon}")

# Сохраняем обновлённый файл
df.to_csv("studios_with_coords.csv", index=False)
print("\n Файл studios_with_coords.csv обновлён")

# Проверяем, нет ли пустых
print(f"\n Пустых координат осталось: {df['lat'].isna().sum()}")

# Показываем результат
print("\n Итоговая таблица:")
print(df[["name", "address", "lat", "lon"]])

Обновлено: Петровский бульвар, 9 ст2... → 55.768588, 37.614779
Обновлено: Красный Октябрь, Болотная набережная, 3 ... → 55.74, 37.609505
Обновлено: Нижний Сусальный переулок, 5 ст4... → 55.760206, 37.663889
Обновлено: Улица Земляной Вал, 50а ст3... → 55.751148, 37.655248
Обновлено: Рождественский бульвар, 10/7 ст1... → 55.766223, 37.626196
Обновлено: Нижняя Сыромятническая улица, 10 ст40... → 55.752794, 37.669782

 Файл studios_with_coords.csv обновлён

 Пустых координат осталось: 0

 Итоговая таблица:
                                 name  \
0                        Grusha Music   
1                         Moon dragon   
2                     KomorKing sound   
3   Продюсерский центр Игоря Сандлера   
4                   Pioneer DJ School   
5                            Хендрикс   
6                         ЦАО Records   
7                         Звук Во рту   
8                        Вайб Рекордс   
9                           333 sound   
10                           Overtime   


Медленно, но верно, мы подходим к ключевому этапу нашей работы. Код создаёт интерактивную карту Москвы с отмеченными на ней студиями звукозаписи. Он загружает файл studios_with_coords.csv, где уже есть все названия, адреса и координаты студий. Затем с помощью библиотеки folium создаёт карту с центром в центре Москвы и масштабом 13 (чтобы были видны районы). Для каждой студии из таблицы код добавляет на карту значок в точку с координатами lat и lon. При клике на значок всплывает окошко с названием студии и адресом. После добавления всех 15 точек код показывает карту в Colabе для удобства, а затем сохраняет карту в файл studios_map.html и скачивает его на компьютер.

In [5]:
import pandas as pd
import folium
from google.colab import files
from IPython.display import display

# Загружаем файл
df = pd.read_csv("studios_with_coords.csv")

# Создаём карту
moscow_map = folium.Map(location=[55.7558, 37.6173], zoom_start=13)

# Добавляем все точки
for _, row in df.iterrows():
    folium.Marker(
        location=[row["lat"], row["lon"]],
        popup=f"<b>{row['name']}</b><br>{row['address']}",
        icon=folium.Icon(color="red", icon="music", prefix="fa")
    ).add_to(moscow_map)

display(moscow_map)

# Сохраняем и скачиваем
moscow_map.save("studios_map.html")
files.download("studios_map.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Я решила расширить предыдущую карту и вручную отобрала ключевые точки на Яндекс.Картах, а если быть точнее, два главных музыкальных вуза, один популярный клуб и два творческих кластера. Код загружает таблицу со студиями, а затем вручную задаётся список из пяти музыкальных локаций: два музыкальных вуза (Гнесинка и Консерватория), один концертный клуб («16 тонн») и два творческих кластера (Artplay и Винзавод). Благодаря этому на одной карте видно и расположение существующих студий, и места скопления музыкантов.



In [14]:
import folium
from IPython.display import display
from google.colab import files

df = pd.read_csv("studios_with_coords.csv")

# Музыкальные места
music_places = [
    ("РАМ им. Гнесиных", 55.7560, 37.5930, "вуз"),
    ("Московская консерватория", 55.7565, 37.6020, "вуз"),
    ("Клуб 16 тонн", 55.7610, 37.5730, "клуб"),
    ("Artplay", 55.7520, 37.6700, "кластер"),
    ("Винзавод", 55.7500, 37.6500, "кластер"),
]

# Цвета по типам
color_by_type = {"вуз": "green", "клуб": "purple", "кластер": "blue"}

# Карта
moscow_map = folium.Map(location=[55.7558, 37.6173], zoom_start=13)

# Студии (они у нас красные)
for _, row in df.iterrows():
    folium.Marker(
        location=[row["lat"], row["lon"]],
        popup=f"<b>🎙️ {row['name']}</b><br>{row['address']}",
        icon=folium.Icon(color="red", icon="music", prefix="fa")
    ).add_to(moscow_map)

# Музыкальные места
for name, lat, lon, place_type in music_places:
    folium.Marker(
        location=[lat, lon],
        popup=f"<b>{name}</b><br>({place_type})",
        icon=folium.Icon(color=color_by_type.get(place_type, "gray"), icon="info-sign")
    ).add_to(moscow_map)

display(moscow_map)
moscow_map.save("studios_with_music.html")
files.download("studios_with_music.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Тепловая карта (уже для второй итерации)

Данная карта будет показывать плотность студий: красные зоны - много конкурентов, синие - мало, там и будем размещать.


In [ ]:
#пупупу